# Mobility-Informed Renewal Equation Framework — Demo Notebook

**Reference:** Mills, C. (2026). *Reproduction numbers for epidemics on human mobility networks.*
Department of Statistics and Pandemic Sciences Institute, University of Oxford.

This notebook walks through the full framework step-by-step using a synthetic 8-node megacity network.
Each section links explicitly to the equations and Table 1–2 quantities from the manuscript.

**Contents**
1. [Setup & infectiousness profile](#1-setup)
2. [Build synthetic mobility network](#2-network)
3. [Run simulation (PDE + calibration)](#3-simulate)
4. [Network-level R(t): threshold, reactivity, risk-averse E(t)](#4-network-R)
5. [Location-level R: outward, inward, meeting](#5-location-R)
6. [Pairwise R matrix and elasticity](#6-pairwise)
7. [Generation time distributions](#7-gt)
8. [Transience: mixing ratio, amplification envelope, condition number](#8-transience)
9. [Type reproduction numbers](#9-type-R)
10. [Spatial source-sink and controllability](#10-control)
11. [Comparing independent estimator bias](#11-bias)
12. [Counterfactual: sparse rural network](#12-counterfactual)


## 1. Setup and infectiousness profile <a id="1-setup"></a>

We model a SARS-CoV-2-like pathogen with an infectiousness profile from
Hart et al. (2022) *Lancet Infect Dis*: Gamma(mean=5.5d, sd=1.8d), truncated at 25 days.

This profile `p(a_E)` is the key biological input; it controls the generation time
distribution **g** and the integral that enters every reproduction number formula (Eq. 7).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mobility_rt_framework import (
    MobilityRtFramework,
    discretise_gamma,
    build_synthetic_network,
    run_all_tests,
    spectral_analysis,
    sensitivity_elasticity,
    type_R_all,
    group_type_R,
    amplification_envelope,
    source_sink_analysis,
    compute_generation_times,
    estimate_R_independent,
    R_network, R_outward, R_inward,
    OKABE_ITO,
)

# Confirm all unit tests pass before proceeding
ok = run_all_tests(verbose=False)
print(f"All validation tests passed: {ok}")


In [ ]:
# Infectiousness profile p(a_E): Gamma(mean=5.5d, sd=1.8d), 25 days
#   Hart WS et al. 2022 Lancet Infect Dis 22(5):603-610
max_days = 25
p = discretise_gamma(mean=5.5, sd=1.8, max_days=max_days)

mean_gt = float(np.sum(np.arange(max_days) * p))
print(f"Mean generation time: {mean_gt:.2f} days")
print(f"p sums to: {p.sum():.8f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(np.arange(max_days), p, color=OKABE_ITO[0], edgecolor="none", width=0.85)
ax.set_xlabel("Infection age $a_E$ (days)")
ax.set_ylabel("$p(a_E)$")
ax.set_title(f"Infectiousness profile — mean {mean_gt:.1f}d (Hart et al. 2022)")
plt.tight_layout(); plt.show()


## 2. Build synthetic mobility network <a id="2-network"></a>

We simulate an 8-node megacity with:
- **2 core/hub nodes** — dense population, 40% commuting fraction
- **3 dense nodes** — medium-high population, 35% commuting
- **2 suburban nodes** — moderate population, 28% commuting  
- **1 peripheral node** — low population, 18% commuting

The mobility matrix **f^{jk}(t)** (Eq. 6) is built from exponential distance
decay + gravity population weighting + day-of-week scaling + daily lognormal noise:

$$f^{jj}(t) = 1 - c_j \quad\text{(home)}, \qquad f^{jk}(t) \propto e^{-d_{jk}/\delta} \cdot N_k^p \quad (j\neq k)$$

**Key property:** `f_jk[t, j, :].sum() == 1` for all t, j (row-stochastic).


In [ ]:
N = 8   # locations
T = 250 # simulation days

node_types = ["core","core","dense","dense","dense","suburban","suburban","peripheral"]
pop_map    = {"core":900_000,"dense":650_000,"suburban":350_000,"peripheral":150_000}
populations = np.array([pop_map[t] for t in node_types], float)
populations *= np.exp(np.random.default_rng(42).normal(0, 0.15, N))
populations = populations.round()

f_jk, populations, node_types = build_synthetic_network(
    N              = N,
    node_types     = node_types,
    populations    = populations,
    T              = T,
    seed           = 42,
    day_variation_sd      = 0.12,
    hub_attraction_power  = 0.5,
    decay_scale           = 20.0,
)

print("f_jk shape:", f_jk.shape)
print("Row sums (t=0):", f_jk[0].sum(axis=1).round(6))   # must all be 1
print("\nPopulations (x1000):", (populations/1e3).round(1))
print("Node types:         ", node_types)
print(f"Total population: {populations.sum()/1e6:.2f} M")


In [ ]:
# Visualise mean mobility matrix and home fractions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

f_mean = f_jk.mean(axis=0)
loc    = [f"L{i+1}" for i in range(N)]

# Mean mobility matrix
im0 = axes[0].imshow(f_mean, cmap="Blues", aspect="auto")
axes[0].set_xticks(range(N)); axes[0].set_xticklabels(loc, fontsize=8, rotation=45)
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(loc, fontsize=8)
axes[0].set_title(r"Mean mobility matrix $\bar{f}^{jk}$", fontsize=10)
axes[0].set_xlabel("Activity $k$"); axes[0].set_ylabel("Residence $j$")
plt.colorbar(im0, ax=axes[0]).set_label(r"$\bar{f}^{jk}$")

# Home fractions over time
f_home = np.array([[f_jk[t,j,j] for j in range(N)] for t in range(T)])
im1 = axes[1].imshow(f_home.T, aspect="auto", cmap="RdYlGn", vmin=0.3, vmax=1.0,
                      extent=[-0.5, T-0.5, N-0.5, -0.5])
axes[1].set_yticks(range(N)); axes[1].set_yticklabels(loc, fontsize=8)
axes[1].set_xlabel("Day $t$"); axes[1].set_ylabel("Residence $j$")
axes[1].set_title(r"Home fraction $f^{jj}(t)$ (day-of-week cycling)", fontsize=10)
plt.colorbar(im1, ax=axes[1]).set_label(r"$f^{jj}(t)$")

plt.tight_layout(); plt.show()
print("Day-of-week effect visible as weekly stripes — Fridays/weekends show higher home fractions")


## 3. Run the simulation <a id="3-simulate"></a>

### What the framework does

1. **Calibration**: scales contact rates so that the initial network reproduction
   number `R(0) = R₀_target` (here 1.5).

2. **PDE stepping** (Eq. 4-6): each day, the infection-age distribution `E^j(t, a_E)`
   is advanced by one step, and new infections are computed from the boundary condition:
   $$E^j(t,0) = S_j(t) \sum_k K_{\text{base}}[k,j] \sum_{a_E} p(a_E)\, E^k(t, a_E)$$

3. **All output quantities** are computed at every time step and returned as arrays.

### Key inputs
| Parameter | Value | Meaning |
|-----------|-------|---------|
| `contact_rate_home` | 13.0 | λ_W: contacts/day at home (POLYMOD, Mossong 2008) |
| `contact_rate_away` | 3.9 | λ_B = 0.30 × λ_W: contacts/day away |
| `R0_target` | 1.5 | Calibration target for R(0) |
| `initial_infections` | 10 in L1 | Single-location seeding |


In [ ]:
# Seed: 10 infections in location 1 (a core/hub node)
initial = np.zeros(N); initial[0] = 10.0

model = MobilityRtFramework(
    f_jk                  = f_jk,
    populations           = populations,
    infectiousness_profile = p,
    contact_rate_home     = 13.0,   # lambda_W (POLYMOD, Mossong 2008)
    contact_rate_away     = 3.9,    # lambda_B = 0.30 * lambda_W
    R0_target             = 1.5,
    initial_infections    = initial,
    T                     = T,
    location_names        = [f"L{i+1} ({t})" for i,t in enumerate(node_types)],
    verbose               = True,
)

results = model.simulate()

# Quick summary
inc   = results["incidence"]
peak  = int(inc.sum(axis=1).argmax())
att   = inc.sum() / populations.sum() * 100
R0_ok = results["R0_achieved"]
print(f"\nR₀ achieved: {R0_ok:.4f}")
print(f"Epidemic peak: day {peak}")
print(f"Attack rate:   {att:.1f}%")
print(f"Output keys: {list(results.keys())}")


## 4. Network-level R(t): threshold, reactivity, risk-averse E(t) <a id="4-network-R"></a>

### Hierarchy of network-level indicators (Table 2)

| Quantity | Definition | Property |
|----------|------------|----------|
| **R(t)** = ρ(**R**(t)) | Spectral radius of NGM (Eq. 23) | **Threshold**: R > 1 ↔ epidemic growth |
| **σ(t)** | Largest singular value of **R**(t) (Eq. 37) | σ ≥ R; σ > 1 and R < 1 → false-action zone |
| **E(t)** | Σ_j [R^j_out]² / Σ_k R^k_out (Eq. 42) | Spatial risk-averse; E > 1 and R < 1 → localised resurgence risk |
| **A₁(1)** | max_k R^k_out | Upper bound on R(t); first-generation epidemicity |

**False-action zone**: when σ(t) > 1 yet R(t) < 1, per-generation incidence can
transiently *amplify* even though the epidemic is asymptotically declining.
This happens when R(t) is highly non-normal (asymmetric mobility).


In [ ]:
t_arr = np.arange(T)
Rn    = results["R_network"]
sig   = results["reactivity"]
Et    = results["E_risk_averse"]
A1_1  = results["R_outward"].max(axis=1)   # A₁(1) = max_k R^k_out (Eq. 40)
OK    = OKABE_ITO

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: full trajectory
ax = axes[0]
ax.plot(t_arr, Rn,   color=OK[4], lw=1.3, label=r"$\mathcal{R}(t)$ (spectral radius)")
ax.plot(t_arr, sig,  color=OK[5], lw=1.1, ls="--", label=r"$\sigma(t)$ (reactivity)")
ax.plot(t_arr, Et,   color=OK[6], lw=1.0, ls=":",  label=r"$\mathcal{E}(t)$ (risk-averse)")
ax.plot(t_arr, A1_1, color=OK[1], lw=0.9, ls="-.", label=r"$\mathcal{A}_1(1)=\max_k R^k_{\rm out}$")
ax.axhline(1.0, color="#666", ls="--", lw=0.9, label="Threshold = 1")
tmask = (sig > 1) & (Rn < 1)
if tmask.any():
    ax.fill_between(t_arr, 1.0, sig, where=tmask, color="orange", alpha=0.25,
                    label=r"False-action zone ($\sigma>1$, $\mathcal{R}<1$)")
ax.set_xlabel("Day $t$", fontsize=11); ax.set_ylabel("Value", fontsize=11)
ax.set_title("Network reproduction numbers over time", fontsize=10)
ax.legend(fontsize=7.5, ncol=2, loc="upper right")

# Right: twin-axis with incidence
ax  = axes[1]
ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
ax.plot(t_arr, Rn, color=OK[4], lw=1.3, label=r"$\mathcal{R}(t)$")
ax.axhline(1.0, color="#666", ls="--", lw=0.9)
ax2.fill_between(t_arr, inc.sum(axis=1)/1e3, alpha=0.2, color=OK[1])
ax2.plot(t_arr, inc.sum(axis=1)/1e3, color=OK[1], lw=1.1, label="Total incidence")
ax.set_xlabel("Day $t$", fontsize=11)
ax.set_ylabel(r"$\mathcal{R}(t)$", color=OK[4], fontsize=11)
ax2.set_ylabel("Incidence ($10^3$)", color=OK[1], fontsize=11)
ax.set_title("R(t) tracks the epidemic curve", fontsize=10)

plt.tight_layout(); plt.show()

print(f"\nR(t) range: [{Rn.min():.3f}, {Rn.max():.3f}]")
print(f"σ(t) range: [{sig.min():.3f}, {sig.max():.3f}]")
print(f"Days with σ>1 and R<1 (false-action): {tmask.sum()}")
print(f"\nInterpretation:")
print(f"  R > 1 (epidemic growing): days 0–{(Rn > 1).sum()-1}")
print(f"  R(t) at peak (day {peak}): {Rn[peak]:.3f}")
print(f"  E(t) > R(t) throughout: {(Et > Rn + 0.01).sum()} days")
print(f"  (E(t) > R(t) → heterogeneous resurgence risk)")


## 5. Location-level reproduction numbers <a id="5-location-R"></a>

### Outward R^k_out(t) and Inward R^j_in(t)

**R^k_out(t)** (Eq. 15) = expected infections generated by an infected resident of k in any location.
Prioritise *stay-at-home / movement restrictions* for high R^k_out locations.

**R^j_in(t)** (Eq. 44) = expected infections received in j from one infected seeded in each location.
Target *travel restrictions / quarantine* for high R^j_in locations.

Neither has the threshold property — that belongs to R(t) = ρ(**R**) alone.
They do track infection source/sink pressure instantaneously.

**Key insight**: R^j_in and R^j_out differ from the closed-population estimator
R̂^j_ind because they capture import/export dynamics that R̂^j_ind ignores.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

Ro = results["R_outward"]   # (T, N)
Ri = results["R_inward"]    # (T, N)
loc = [f"L{i+1}" for i in range(N)]

# R_outward
pos = Ro[Ro > 0]
im0 = axes[0].imshow(Ro.T, aspect="auto", cmap="plasma", origin="upper",
                      extent=[-0.5,T-0.5,N-0.5,-0.5],
                      vmin=np.percentile(pos,2), vmax=np.percentile(pos,97))
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(loc, fontsize=8)
axes[0].set_xlabel("Day $t$"); axes[0].set_ylabel("Infector $k$")
axes[0].set_title(r"$R^k_{\rm out}(t)$: outward R", fontsize=10)
plt.colorbar(im0, ax=axes[0])

# R_inward
pos2 = Ri[Ri > 0]
im1 = axes[1].imshow(Ri.T, aspect="auto", cmap="viridis", origin="upper",
                      extent=[-0.5,T-0.5,N-0.5,-0.5],
                      vmin=np.percentile(pos2,2), vmax=np.percentile(pos2,97))
axes[1].set_yticks(range(N)); axes[1].set_yticklabels(loc, fontsize=8)
axes[1].set_xlabel("Day $t$"); axes[1].set_ylabel("Infectee $j$")
axes[1].set_title(r"$R^j_{\rm in}(t)$: inward R", fontsize=10)
plt.colorbar(im1, ax=axes[1])

# R_meeting
Rm = results["R_meeting"]
posm = Rm[Rm>0]
im2 = axes[2].imshow(Rm.T, aspect="auto", cmap="cividis", origin="upper",
                      extent=[-0.5,T-0.5,N-0.5,-0.5],
                      vmin=np.percentile(posm,2), vmax=np.percentile(posm,97))
axes[2].set_yticks(range(N)); axes[2].set_yticklabels(loc, fontsize=8)
axes[2].set_xlabel("Day $t$"); axes[2].set_ylabel("Meeting location $l$")
axes[2].set_title(r"$R^l_{\rm meeting}(t)$: meeting-location R", fontsize=10)
plt.colorbar(im2, ax=axes[2])

plt.tight_layout(); plt.show()

print("Outward R at peak day (sorted, high to low):")
for j in np.argsort(Ro[peak])[::-1]:
    print(f"  {loc[j]:20s}: R_out={Ro[peak,j]:.3f},  R_in={Ri[peak,j]:.3f}")


## 6. Pairwise R matrix and elasticity <a id="6-pairwise"></a>

### Pairwise R^{kj}(t)

**R^{kj}(t)** (Eq. 14) = expected infections generated in residents of j by residents of k.
Identifies specific transmission corridors — where to target mobility-corridor interventions.

### Sensitivity and elasticity (Eqs. 29-35)

**s^{kj}(t)** = ∂R(t)/∂R^{kj}(t) = v_j × v*_k / (v·v*)

**ε^{kj}(t)** = (R^{kj}/R) × s^{kj}  → "1% change in R^{kj} produces ε^{kj}% change in R(t)"

Elasticities sum to exactly 1: Σ_{k,j} ε^{kj}(t) = 1.
Corridors with high ε^{kj}(t) are where disease control gives the largest network-level benefit.

**ε^k_out** = Σ_j ε^{kj}: ranks *infector source locations* for intervention.
**ε^j_in** = Σ_k ε^{kj}: ranks *infectee destination locations* for protection.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

Rpk  = results["R_pairwise"][peak]   # R^{kj} at epidemic peak
epk  = results["elasticity"][peak]   # ε^{kj} at epidemic peak
Epk  = results["incidence_matrix"][peak]  # pairwise incidence

# Pairwise R at peak
im0 = axes[0].imshow(Rpk, cmap="PuOr", aspect="auto")
axes[0].set_xticks(range(N)); axes[0].set_xticklabels(loc, fontsize=6, rotation=45)
axes[0].set_yticks(range(N)); axes[0].set_yticklabels(loc, fontsize=6)
axes[0].set_xlabel("Infectee $j$"); axes[0].set_ylabel("Infector $k$")
axes[0].set_title(rf"$R^{{kj}}$ at day {peak}", fontsize=10)
plt.colorbar(im0, ax=axes[0])

# Elasticity at peak
im1 = axes[1].imshow(epk, cmap="YlOrRd", aspect="auto", vmin=0)
axes[1].set_xticks(range(N)); axes[1].set_xticklabels(loc, fontsize=6, rotation=45)
axes[1].set_yticks(range(N)); axes[1].set_yticklabels(loc, fontsize=6)
axes[1].set_xlabel("Infectee $j$"); axes[1].set_ylabel("Infector $k$")
axes[1].set_title(rf"$\varepsilon^{{kj}}$ elasticity at day {peak}", fontsize=10)
plt.colorbar(im1, ax=axes[1])
axes[1].text(0.02,0.02,f"sum={epk.sum():.6f}",transform=axes[1].transAxes,fontsize=8,va="bottom")

# Infector vs infectee elasticity scatter (time-mean)
me_out = results["elasticity_out"].mean(axis=0)
me_in  = results["elasticity_in"].mean(axis=0)
sc = axes[2].scatter(me_out, me_in, s=80, c=np.arange(N), cmap="tab10",
                      edgecolors="#222", linewidths=0.6)
for j in range(N):
    axes[2].annotate(f"L{j+1}", (me_out[j], me_in[j]),
                     textcoords="offset points", xytext=(5,3), fontsize=8)
mx = max(me_out.max(), me_in.max()) * 1.05
axes[2].plot([0, mx],[0, mx], color="#aaa", ls="--", lw=0.9, label="Equal line")
axes[2].set_xlabel(r"Time-mean $\varepsilon^k_{\rm out}$", fontsize=10)
axes[2].set_ylabel(r"Time-mean $\varepsilon^j_{\rm in}$", fontsize=10)
axes[2].set_title("Elasticity: infector vs infectee\n(above line → net importer)", fontsize=9)
axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f"\nTop 3 infector locations by ε^k_out at peak:")
for j in np.argsort(results['elasticity_out'][peak])[::-1][:3]:
    print(f"  {loc[j]:22s}: ε_out={results['elasticity_out'][peak,j]:.4f}")
print(f"\nElasticity sum at peak: {results['elasticity'][peak].sum():.10f}  (must = 1.0)")

## 7. Generation time distributions <a id="7-gt"></a>

### The universal GT property (key result)

Under the separable kernel assumption (Eq. 7):
$$\lambda^{kl}_E(t, a_E) = \chi^{kl}(t) \cdot p(a_E) / N^l_{\rm eff}(t)$$

the generation time distribution is the **same for all pairwise (k,j) combinations
and invariant to calendar time t**:
$$g^{kj}(t, a_E) = \frac{K^{kj}(t, a_E)}{R^{kj}(t)} = \frac{p(a_E) \cdot S_j \cdot K_{\rm base}[k,j]}{S_j \cdot K_{\rm base}[k,j] \cdot \int p} = p(a_E)$$

This justifies the common epidemiological assumption of a time-invariant generation
time distribution, and confirms it emerges from mechanistic first principles here.

The *network-level* GT g_network(t, a_E) (Eq. 27) weights pairwise GTs by the
eigenvector structure — it also equals p(a_E) for separable kernels.


In [ ]:
days_arr = np.arange(max_days)
gt_peak = model.get_generation_times_snapshot(peak)

print("Universal GT property verification:")
print(f"  g_universal sums to: {gt_peak['g_universal'].sum():.10f}")
print(f"  All g_pairwise[:, k, j] == p: {np.allclose(gt_peak['g_pairwise'], p[:, None, None], atol=1e-12)}")
print(f"  g_network == p: {np.allclose(gt_peak['g_network'], p, atol=1e-12)}")
print(f"  Mean generation time: {gt_peak['mean_gt']:.2f} days")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, t_idx, label in zip(axes, 
                              [max(1, peak//3), peak, min(T-1, peak+30)],
                              ["Early", "Peak", "Late"]):
    gt = model.get_generation_times_snapshot(t_idx)
    ax.bar(days_arr, gt["g_universal"], color=OKABE_ITO[0], alpha=0.7, 
           label=f"p(a_E) (mean={gt['mean_gt']:.1f}d)")
    ax.set_xlabel("$a_E$ (days)"); ax.set_ylabel("Probability")
    ax.set_title(f"{label} (day {t_idx}): all GT types coincide\n(universal property)", fontsize=8.5)
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print("\nConclusion: GT is universal — no difference between pairwise, outward, inward, or network GT.")
print("This arises from the separable kernel structure and is justified by mechanistic first principles.")


## 8. Transience: mixing ratio, amplification, condition number <a id="8-transience"></a>

### Short-term indicators (Table 2, Section 3.1.5)

R(t) measures *asymptotic* per-generation growth — but short-term dynamics can
differ markedly, especially with heterogeneous mobility.

| Quantity | Formula | Meaning |
|----------|---------|---------|
| **s(t)** | \|λ₂\|/R(t) | Mixing ratio: how many generations to reach stable v* |
| **ζ(t)** | R(t) − \|λ₂\| | Spectral gap: larger → faster convergence to v* |
| **σ(t)** | \|\|**R**\|\|₂ | Reactivity: max one-generation amplification |
| **A(n)** | \|\|**R**^n\|\|₂ | ℓ² amplification envelope over n generations |
| **A₁(n)** | \|\|**R**^n\|\|₁ | ℓ¹ envelope = worst single-location seeding |
| **κ(R)** | \|\|v\|\|₂\|\|v*\|\|₂/\|v·v*\| | Condition number: sensitivity of R(t) to noise |


In [ ]:
early_t = max(1, peak // 3)
late_t  = min(T-1, peak + 30)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Mixing ratio and reactivity
ax = axes[0]
mix = results["mixing_ratio"]
sig = results["reactivity"]
ax.plot(t_arr, mix, color=OKABE_ITO[2], lw=1.2, label=r"$s(t)=|\lambda_2|/\mathcal{R}$")
ax.set_ylim(0, 1.05)
ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
ax2.plot(t_arr, sig, color=OKABE_ITO[5], lw=1.1, ls="--", label=r"$\sigma(t)$")
ax2.axhline(1.0, color="#888", ls=":", lw=0.8)
ax.set_xlabel("Day $t$"); ax.set_ylabel(r"Mixing ratio $s(t)$", color=OKABE_ITO[2])
ax2.set_ylabel(r"Reactivity $\sigma(t)$", color=OKABE_ITO[5])
ax.tick_params(axis="y", labelcolor=OKABE_ITO[2])
ax2.tick_params(axis="y", labelcolor=OKABE_ITO[5])
h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, fontsize=7.5, loc="upper right")
ax.set_title("Mixing ratio and reactivity", fontsize=9)

# Amplification envelopes
ax = axes[1]
n_max = 20
for t_phase, name, col in [(early_t,"early",OKABE_ITO[2]),
                             (peak,"peak",OKABE_ITO[5]),
                             (late_t,"late",OKABE_ITO[0])]:
    env = model.get_amplification_envelope(t_phase, n_max=n_max)
    rn  = env["rho_n"]
    ax.plot(env["n"], env["A"]  / (rn + 1e-300), color=col, lw=1.1,
            label=f"A(n) {name}")
    ax.plot(env["n"], env["A1"] / (rn + 1e-300), color=col, lw=0.7, ls=":")
ax.axhline(1.0, color="#888", ls="--", lw=0.8)
ax.set_xlabel("$n$ (generations)")
ax.set_ylabel(r"Envelope / $\mathcal{R}^n$")
ax.set_title(r"Amplification: $A(n)/\mathcal{R}^n$ (solid), $\mathcal{A}_1(n)/\mathcal{R}^n$ (dot)", fontsize=8.5)
ax.legend(fontsize=7, ncol=2)

# Condition number
ax = axes[2]
cond_clip = np.minimum(results["condition_number"], 5e3)
ax.semilogy(t_arr, cond_clip, color=OKABE_ITO[1], lw=1.1)
ax.axvline(peak, color="#777", ls=":", lw=0.8, label=f"Peak (day {peak})")
ax.set_xlabel("Day $t$"); ax.set_ylabel(r"$\kappa(\mathbf{R}(t))$ [log]")
ax.set_title("Condition number\n(sensitivity of R to mobility noise)", fontsize=9)
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f"Mixing ratio range: [{mix.min():.3f}, {mix.max():.3f}]")
print(f"s close to 1 → slow mixing (spatial distribution slow to stabilise)")
print(f"s close to 0 → fast mixing (rapid convergence to stable distribution v*)")


## 9. Type reproduction numbers <a id="9-type-R"></a>

### T^j_type(t): does location j alone sustain the epidemic? (Eq. 53)

$$T^j_{\rm type}(t) = R_{jj} + R_{jJ}(I - R_{JJ})^{-1}R_{Jj}$$

where J = all locations except j.

**Key properties:**
- T^j_type(t) > 1 ↔ R(t) > 1 for irreducible NGM (Eq. 54)
- Undefined (→ ∞) when ρ(R_JJ) ≥ 1 — the background network sustains the epidemic on its own
- A minimal fractional reduction c_j > 1 − 1/T^j_type is sufficient to eliminate the location-j cycle

**Group type R^P_type(t)** (Eq. 55): extend to a *set* of locations P to overcome the
single-location undefinedness problem.


In [ ]:
# Compute type R at epidemic peak (expensive for large N)
TR_peak = type_R_all(results["R_pairwise"][peak])
print(f"Type R numbers at peak (day {peak}):")
print(f"  R(t) = rho = {results['R_network'][peak]:.4f}")
for j in range(N):
    val = TR_peak[j]
    status = "defined" if not np.isnan(val) else "undefined (background self-sustaining)"
    val_str = f"{val:.4f}" if not np.isnan(val) else "NaN"
    print(f"  {loc[j]:22s}: T^j_type = {val_str:>8}  ({status})")

# Group type R for sets of locations
print("\nGroup type R^P_type at peak:")
for P_desc, P in [
    ("All cores (L1,L2)",        [0, 1]),
    ("Cores+dense (L1-L5)",      [0, 1, 2, 3, 4]),
    ("All locations",            list(range(N))),
]:
    gtr = group_type_R(results["R_pairwise"][peak], P)
    gtr_str = f"{gtr:.4f}" if not np.isnan(gtr) else "NaN"
    print(f"  P={P_desc:25s}: T^P_type = {gtr_str}")

print("\nNote: T^j_type is undefined when the background network (J = all \\ j)")
print("can sustain the epidemic alone — this is mathematically correct, not a bug.")
print("Use group T^P_type with a larger P to handle this.")

## 10. Spatial source-sink and controllability <a id="10-control"></a>

### Source-sink analysis (Eq. 45)

A location is a **source** if R^k_out > R^k_in (net exporter of infections)
and a **sink** if R^k_out < R^k_in (net importer).

The ratio η(t) = CV(R^k_out) / CV(R^j_in) tells us whether heterogeneity in
outward or inward transmission is larger, guiding whether to target source
control or destination protection.

### Minimum control effort

To bring R(t) below 1, we need to reduce between-location transmission by
at least u_homogeneous = 1 − 1/R(t) uniformly.

Targeted reductions prioritised by elasticity can achieve the same result
with less total effort by focusing on the highest-impact corridors.


In [ ]:
# Source-sink at peak
ss = model.source_sink_at(peak)
print(f"Source-sink analysis at peak (day {peak}):")
print(f"  π_within = {ss['pi_within']:.3f}  (fraction of infections within-location)")
print(f"  π_between = {ss['pi_between']:.3f}  (fraction between-location)")
print(f"  η = CV(R_out)/CV(R_in) = {ss['eta']:.3f}")
print(f"  η > 1 → target source control; η < 1 → target destination protection\n")
print("  Sources (net exporters):", [loc[j] for j in ss["sources"]])
print("  Sinks   (net importers):", [loc[j] for j in ss["sinks"]])

# Controllability
ctrl = model.controllability_effort(peak)
print(f"\nControllability at peak (R={ctrl['rho']:.3f}):")
print(f"  Homogeneous reduction needed everywhere: {ctrl['u_homogeneous']*100:.1f}%")
print(f"  Prioritised reductions (by elasticity):")
for idx in ctrl["priority_order"][:N]:
    if ctrl["u_heterogeneous"][idx] > 1e-6:
        print(f"    {loc[idx]:22s}: {ctrl['u_heterogeneous'][idx]*100:.1f}% reduction")

# Visualise
fig, ax = plt.subplots(figsize=(6, 3.5))
net = ss["net_export"]
clrs = [OKABE_ITO[5] if x > 0 else OKABE_ITO[4] for x in net]
ax.barh(range(N), net, color=clrs, height=0.65, edgecolor="none")
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(range(N)); ax.set_yticklabels(loc, fontsize=8)
ax.set_xlabel(r"Net export $R^k_{\rm out} - R^k_{\rm in}$")
ax.set_title(f"Source-sink decomposition at peak (day {peak})", fontsize=9)
from matplotlib.patches import Patch
ax.legend([Patch(fc=OKABE_ITO[5]), Patch(fc=OKABE_ITO[4])],
          ["Source (exports > imports)", "Sink (imports > exports)"], fontsize=8)
plt.tight_layout(); plt.show()


## 11. Comparing independent estimator bias <a id="11-bias"></a>

The independent per-location estimator R̂^j_ind(t) (Eq. 56, Cori 2013)
treats each location as a closed population:
$$\hat{R}^j_{\rm ind}(t) \approx \frac{E^j(t,0)}{\sum_{a_E} p(a_E) E^j(t-a_E, 0)}$$

This **ignores** import and export dynamics and therefore:
- *Over-estimates* R during decline (imported cases inflate the numerator)  
- *Under-estimates* R early on (exported cases are missed)
- Differs most from R^j_out and R^j_in for locations with strong commuting

Comparing R̂^j_ind with R^j_out and R^j_in reveals the **spatial-neglect bias**
that arises from using closed-population methods in mobile populations.


In [ ]:
R_ind = results["R_independent"]   # (T, N)
R_out = results["R_outward"]
R_in  = results["R_inward"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = [OKABE_ITO[0], OKABE_ITO[2], OKABE_ITO[5], OKABE_ITO[4]]

for ax, R_framework, label_fw, ylabel in [
    (axes[0], R_out, r"$R^k_{\rm out}$ (solid)", r"$R$"),
    (axes[1], R_in,  r"$R^j_{\rm in}$ (solid)", r"$R$"),
]:
    for j, col in zip([0, 2, 5, 7], colors):   # hub, dense, suburban, peripheral
        vi = ~np.isnan(R_ind[:, j])
        ax.plot(np.where(vi)[0], R_ind[vi, j], "--", color=col, lw=0.9, alpha=0.7)
        ax.plot(t_arr, R_framework[:, j], "-", color=col, lw=1.0, label=loc[j])
    ax.axhline(1.0, color="#555", ls="--", lw=0.8)
    ax.set_xlabel("Day $t$"); ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(rf"{label_fw} vs $\hat{{R}}^j_{{\rm ind}}$ (dashed)", fontsize=9.5)
    ax.legend(fontsize=7.5, ncol=2)
    ax.set_ylim(0, None)

plt.tight_layout(); plt.show()

# Quantify bias at peak
print("Bias at epidemic peak (R̂_ind - R_out):")
bias_peak = R_ind[peak] - R_out[peak]
for j in range(N):
    if not np.isnan(R_ind[peak, j]):
        print(f"  {loc[j]:22s}: bias = {bias_peak[j]:+.3f}")
print("\nPositive bias → R̂_ind overestimates true outward transmission")
print("Negative bias → R̂_ind underestimates true outward transmission")

## 12. Counterfactual: sparse rural network <a id="12-counterfactual"></a>

We compare our urban megacity (Scenario A) with a sparse rural network (Scenario B)
that has much higher home fractions (residents spend less time away from home).

This illustrates how the framework generalises across different network types
and how mobility patterns shape all the reproduction number quantities.


In [ ]:
# Build sparse rural network
N_B   = N
T_B   = T
types_B = ["capital","peri-capital","rural","rural","rural","rural","remote-rural","remote-rural"]
pop_B   = np.array([3e6, 1.8e6, 0.9e6, 0.9e6, 0.6e6, 0.6e6, 0.4e6, 0.4e6])
cfrac_B = np.array([0.06, 0.08, 0.025, 0.025, 0.015, 0.015, 0.01, 0.01])

f_B, pop_B, types_B = build_synthetic_network(
    N=N_B, node_types=types_B, populations=pop_B, T=T_B,
    commuting_fracs=cfrac_B, decay_scale=200.0, seed=99,
    day_variation_sd=0.10,
)
seed_B = np.zeros(N_B); seed_B[0] = 10.0

model_B = MobilityRtFramework(
    f_jk=f_B, populations=pop_B, infectiousness_profile=p,
    contact_rate_home=13.0, contact_rate_away=3.9, R0_target=1.5,
    initial_infections=seed_B, T=T_B,
    location_names=[f"L{i+1} ({t})" for i,t in enumerate(types_B)],
    verbose=False,
)
res_B = model_B.simulate()

print("Comparison: Dense urban (A) vs Sparse rural (B)")
inc_B = res_B["incidence"]
peak_B = int(inc_B.sum(axis=1).argmax())
print(f"  Scenario A — attack rate: {inc.sum()/populations.sum()*100:.1f}%, peak day: {peak}")
print(f"  Scenario B — attack rate: {inc_B.sum()/pop_B.sum()*100:.1f}%, peak day: {peak_B}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, res, lab, pops in [(axes[0], results, "Dense urban (A)", populations),
                             (axes[1], res_B, "Sparse rural (B)", pop_B)]:
    t_a = np.arange(T)
    Rn_x = res["R_network"]; sg_x = res["reactivity"]; Et_x = res["E_risk_averse"]
    ax.plot(t_a, Rn_x, color=OKABE_ITO[4], lw=1.2, label=r"$\mathcal{R}(t)$")
    ax.plot(t_a, sg_x, color=OKABE_ITO[5], lw=1.0, ls="--", label=r"$\sigma(t)$")
    ax.plot(t_a, Et_x, color=OKABE_ITO[6], lw=0.9, ls=":", label=r"$\mathcal{E}(t)$")
    ax.axhline(1.0, color="#666", ls="--", lw=0.8)
    ax2 = ax.twinx(); ax2.spines["right"].set_visible(True)
    ax2.fill_between(t_a, res["incidence"].sum(axis=1)/pops.sum()*100, alpha=0.15, color=OKABE_ITO[1])
    ax2.set_ylabel("Attack rate (%/day)", color=OKABE_ITO[1], fontsize=9)
    ax.set_xlabel("Day $t$"); ax.set_ylabel(r"$\mathcal{R}, \sigma, \mathcal{E}$", fontsize=10)
    ax.set_title(f"Scenario: {lab}\nσ≈R (near-normal NGM in sparse rural)", fontsize=9)
    ax.legend(fontsize=7, ncol=2, loc="upper right")
plt.tight_layout(); plt.show()

print("\nKey contrast: in the sparse rural network, σ≈R (nearly symmetric mobility)")
print("→ little false-action risk, fast mixing, lower condition number")
print("In the dense urban network, σ > R is larger due to asymmetric hub flows")


## Summary

This notebook demonstrated the full output suite of the `MobilityRtFramework`:

### Framework inputs (what *you* provide)
| Input | Shape | Description |
|-------|-------|-------------|
| `f_jk` | (T,N,N) | Row-stochastic mobility matrix |
| `populations` | (N,) | Resident populations |
| `infectiousness_profile` | (max_days,) | p(a_E), normalised |
| `contact_rate_home` | scalar | λ_W |
| `contact_rate_away` | scalar | λ_B |
| `R0_target` | scalar | Calibration target |
| `initial_infections` | (N,) | Seed vector |

### Framework outputs (what you get)
| Output | Shape | Equation | Description |
|--------|-------|----------|-------------|
| `R_pairwise` | (T,N,N) | Eq. 14 | R^{kj}(t) |
| `R_outward` | (T,N) | Eq. 15 | R^k_out(t) |
| `R_inward` | (T,N) | Eq. 44 | R^j_in(t) |
| `R_network` | (T,) | Eq. 23 | R(t)=ρ(**R**) |
| `R_meeting` | (T,N) | Eq. 49 | R^l_meeting(t) |
| `E_risk_averse` | (T,) | Eq. 42 | E(t) |
| `elasticity` | (T,N,N) | Eq. 33 | ε^{kj}(t) |
| `sensitivity` | (T,N,N) | Eq. 31 | s^{kj}(t) |
| `mixing_ratio` | (T,) | Eq. 36 | s(t) |
| `reactivity` | (T,) | Eq. 37 | σ(t) |
| `condition_number` | (T,) | Eq. 38 | κ(**R**(t)) |
| `reproductive_value` | (T,N) | Eq. 26 | v(t) |
| `stable_distribution` | (T,N) | Eq. 25 | v*(t) |
| `incidence_matrix` | (T,N,N) | Eq. 43 | E^{k→j}(t,0) |

Compute additional quantities on-demand:
```python
type_R_all(results["R_pairwise"][t])               # T^j_type at day t  (Eq. 53)
group_type_R(results["R_pairwise"][t], P=[0,1])    # T^P_type at day t  (Eq. 55)
model.get_amplification_envelope(t, n_max=20)       # A(n), A₁(n)        (Eqs 39-40)
model.source_sink_at(t)                             # source-sink         (Eq. 45)
model.controllability_effort(t)                     # minimum reduction
```
